# 03 Statistical Tests & Effect Sizes
This notebook computes statistical tests (t-test for DTI, Chi-square for Grade/Purpose), effect sizes (Cramérs V, Cohen d), and 95% CIs for default rates.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv('../data/processed/cleaned_loans.csv')
eligible = df[df['outcome_eligible'] == 1]


In [ ]:
# 1. T-test for DTI between Defaulted and Non-Defaulted
dti_default = eligible[eligible['default_flag'] == 1]['debt_to_income'].dropna()
dti_paid = eligible[eligible['default_flag'] == 0]['debt_to_income'].dropna()
t_stat, p_val = stats.ttest_ind(dti_default, dti_paid, equal_var=False)
print(f'T-statistic: {t_stat:.4f}, p-value: {p_val:.4f}')

# Cohen's d effect size for DTI
n1, n2 = len(dti_default), len(dti_paid)
var1, var2 = np.var(dti_default, ddof=1), np.var(dti_paid, ddof=1)
pooled_sd = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
cohens_d = (np.mean(dti_default) - np.mean(dti_paid)) / pooled_sd
print(f\'Cohen\'s d: {cohens_d:.4f}')


In [ ]:
# 2. Chi-square test for Grade
contingency = pd.crosstab(eligible['grade'], eligible['default_flag'])
chi2, p, dof, ex = stats.chi2_contingency(contingency)
print(f'Chi2: {chi2:.4f}, p-value: {p:.4f}')

# Cramér's V effect size
n = contingency.sum().sum()
min_dim = min(contingency.shape) - 1
cramer_v = np.sqrt(chi2 / (n * min_dim))
print(f\'Cramér\'s V: {cramer_v:.4f}')


In [ ]:
# 3. Confidence Intervals for Overall Default Rate (Wald CI)
defaults = eligible['default_flag'].sum()
n = len(eligible)
p = defaults / n
z = 1.96
margin = z * np.sqrt(p * (1 - p) / n)
print(f'Overall Default Rate: {p:.4f}')
print(f'95% CI: [{p - margin:.4f}, {p + margin:.4f}]')
